# MILAN CSV

In [29]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo de gráficos
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

### Leer conjunto de datos y primer vistazo

In [30]:
# Leer el csv y sacar por pantalla las cinco primeras filas.
df = pd.read_csv("data/milan_airbnb.csv")
df.head()

,id,name,host_id,host_name,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,6400,The Studio Milan,13822,Francesca,TIBALDI,45.44119,9.17813,Private room,100,4,12,19/04/10,0.14,1,358
1,23986,""" Characteristic Milanese flat""",95941,Jeremy,NAVIGLI,45.44806,9.17373,Entire home/apt,150,1,15,07/09/20,0.21,1,363
2,28300,nice flat near the park,121663,Marta,SARPI,45.47647,9.17359,Private room,180,1,8,22/04/12,0.11,1,365
3,32119,Nico & Cynthia's Easy Yellow Suite,138683,Nico&Cinzia,VIALE MONZA,45.52014,9.22300,Entire home/apt,75,2,15,01/07/18,0.23,3,200
4,32649,Nico&Cinzia's Red Easy Suite!,138683,Nico&Cinzia,VIALE MONZA,45.51874,9.22495,Entire home/apt,71,2,29,23/10/16,0.71,3,308


### Exploración de datos

In [31]:
# Descripción del conjunto de datos, estándar
df.describe(include='all')

,id,name,host_id,host_name,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
count,1.832200e+04,18312,1.832200e+04,18198,18322,18322.000000,18322.000000,18322,18322.000000,18322.000000,18322.000000,13260,13260.000000,18322.000000,18322.000000
unique,NaN,18050,NaN,2917,87,NaN,NaN,4,NaN,NaN,NaN,2039,NaN,NaN,NaN
top,NaN,Milan Royal Suites - Duomo Bilocale,NaN,Andrea,BUENOS AIRES - VENEZIA,NaN,NaN,Entire home/apt,NaN,NaN,NaN,14/04/19,NaN,NaN,NaN
freq,NaN,5,NaN,430,1373,NaN,NaN,13605,NaN,NaN,NaN,218,NaN,NaN,NaN
mean,2.587132e+07,NaN,8.494918e+07,NaN,NaN,45.471318,9.187382,NaN,115.094913,5.798112,23.720827,NaN,0.799344,14.421897,153.508624
std,1.525744e+07,NaN,1.040579e+08,NaN,NaN,0.020731,0.029543,NaN,290.793019,26.687720,57.657486,NaN,1.220346,46.295635,138.757302
min,6.400000e+03,NaN,1.944000e+03,NaN,NaN,45.395050,9.060680,NaN,8.000000,1.000000,0.000000,NaN,0.010000,1.000000,0.000000
25%,1.214067e+07,NaN,1.243642e+07,NaN,NaN,45.454462,9.168782,NaN,50.000000,1.000000,0.000000,NaN,0.100000,1.000000,0.000000
50%,2.588718e+07,NaN,3.141304e+07,NaN,NaN,45.470875,9.186291,NaN,73.500000,2.000000,3.000000,NaN,0.310000,1.000000,123.000000
75%,3.926758e+07,NaN,1.335227e+08,NaN,NaN,45.486850,9.209680,NaN,110.000000,3.000000,19.000000,NaN,0.990000,4.000000,302.000000


In [32]:
# Información sobre el tipo de datos de cada feature
print("--- Información de tipos de datos ---")
df.info()

--- Información de tipos de datos ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18322 entries, 0 to 18321
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              18322 non-null  int64  
 1   name                            18312 non-null  object 
 2   host_id                         18322 non-null  int64  
 3   host_name                       18198 non-null  object 
 4   neighbourhood                   18322 non-null  object 
 5   latitude                        18322 non-null  float64
 6   longitude                       18322 non-null  float64
 7   room_type                       18322 non-null  object 
 8   price                           18322 non-null  int64  
 9   minimum_nights                  18322 non-null  int64  
 10  number_of_reviews               18322 non-null  int64  
 11  last_review                     13260 non-null  object 

#### Calcular el número de nulos de cada feature

In [33]:
# Contar los nulos por variable
print("--- Valores nulos por columna ---")
null_counts = df.isnull().sum()
print(null_counts)
if null_counts.sum() > 0:
    print("\nColumnas con valores nulos:")
    print(null_counts[null_counts > 0])
else:
    print("✓ No hay valores nulos en el dataset")

--- Valores nulos por columna ---
id                                   0
name                                10
host_id                              0
host_name                          124
neighbourhood                        0
latitude                             0
longitude                            0
room_type                            0
price                                0
minimum_nights                       0
number_of_reviews                    0
last_review                       5062
reviews_per_month                 5062
calculated_host_listings_count       0
availability_365                     0
dtype: int64

Columnas con valores nulos:
name                   10
host_name             124
last_review          5062
reviews_per_month    5062
dtype: int64


#### Buscar valores extraños. Ver los valores únicos en cada feature

In [34]:
unique_values = pd.DataFrame({
    'features': df.columns,
    'n_values': [df[col].unique() for col in df.columns]
})
unique_values
unique_vals = pd.DataFrame({
    'features': df.columns,
    'n_values': [df[col].unique() for col in df.columns]
})
unique_vals

,features,n_values
0,id,"[6400, 23986, 28300, 32119, 32649, 37256, 4047..."
1,name,"[The Studio Milan, "" Characteristic Milanese f..."
2,host_id,"[13822, 95941, 121663, 138683, 119002, 174203,..."
3,host_name,"[Francesca, Jeremy, Marta, Nico&Cinzia, Gianca..."
4,neighbourhood,"[TIBALDI, NAVIGLI, SARPI, VIALE MONZA, BUENOS ..."
5,latitude,"[45.44119, 45.44806, 45.47647, 45.52014, 45.51..."
6,longitude,"[9.17813, 9.17373, 9.17359, 9.223, 9.22495, 9...."
7,room_type,"[Private room, Entire home/apt, Shared room, H..."
8,price,"[100, 150, 180, 75, 71, 55, 199, 76, 145, 80, ..."
9,minimum_nights,"[4, 1, 2, 3, 20, 7, 5, 90, 15, 30, 60, 6, 365,..."


In [35]:
# Obtener dataframe con features y sus valores únicos
unique_values_info = []
for col in df.columns:
    unique_vals = df[col].unique()
    unique_values_info.append({
        'features': col,
        'n_values': len(unique_vals),
        'values': str(list(unique_vals)[:10])  # Mostrar solo los primeros 10
    })

unique_df = pd.DataFrame(unique_values_info)
print("--- Valores únicos por feature ---")
unique_df[['features', 'n_values']]

--- Valores únicos por feature ---


,features,n_values
0,id,18322
1,name,18051
2,host_id,12213
3,host_name,2918
4,neighbourhood,87
5,latitude,7419
6,longitude,9134
7,room_type,4
8,price,465
9,minimum_nights,68


#### Tratar aquellos valores que entendamos que sean nulos

In [36]:
# Buscar valores que podrían representar nulos (como '?')
print("--- Verificando valores extraños ---")
strange_values_found = False
for col in df.columns:
    unique_vals = df[col].unique()
    if '?' in unique_vals:
        count_question = (df[col] == '?').sum()
        print(f" '{col}' contiene {count_question} valores '?' (posibles nulos)")
        strange_values_found = True

if not strange_values_found:
    print("✓ No se encontraron valores extraños aparentes")
    # Imputaciones: reemplazar '?' por moda o eliminar filas
    print("--- Tratamiento de valores faltantes ---")

    original_shape = df.shape
    
for col in df.columns:
    if '?' in df[col].values:
        print(f"Tratando '{col}'...")
        # Opción 1: Imputar con la moda
        df[col] = df[col].replace('?', np.nan)
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)
        print(f"  → Imputado con moda: {mode_value}")

print(f"✓ Dataset shape: {original_shape} → {df.shape}")


--- Verificando valores extraños ---
✓ No se encontraron valores extraños aparentes
--- Tratamiento de valores faltantes ---
✓ Dataset shape: (18322, 15) → (18322, 15)


#### Mirad cuántos valores hay en cada feature, ¿Todas las features aportan información? Si alguna no aporta información, eliminadla

In [37]:
# Eliminar features que no aportan información (solo un valor único)

print("--- Verificando features informativas ---")
single_value_cols = []
for col in df.columns:
    unique_count = df[col].nunique()
    if unique_count <= 1:
        single_value_cols.append(col)
        print(f"⚠️  '{col}' tiene solo {unique_count} valor único")

if single_value_cols:
    df = df.drop(columns=single_value_cols)
    print(f"✓ Eliminadas {len(single_value_cols)} columnas sin información")
else:
    print("✓ Todas las features aportan información")

print(f"Forma final del dataset: {df.shape}")

--- Verificando features informativas ---
✓ Todas las features aportan información
Forma final del dataset: (18322, 15)


#### Separar entre variables predictoras y variables a predecir

In [40]:
# La variable que trata de predecir este conjunto de datos es 'class' (poisonous)
target_col = 'price' if 'price' in df.columns else df.columns[0]
y = df[target_col]
X = df.drop(columns=[target_col])

print(f"Variable objetivo: {target_col}")
print(f"Variables predictoras: {X.shape[1]} features")
print(f"\n--- Distribución de la variable objetivo ---")
print(y.value_counts())
print(f"\nPorcentajes:")
print(y.value_counts(normalize=True) * 100)

Variable objetivo: price
Variables predictoras: 14 features

--- Distribución de la variable objetivo ---
price
50     840
60     726
80     703
70     691
100    652
      ... 
232      1
531      1
730      1
467      1
422      1
Name: count, Length: 465, dtype: int64

Porcentajes:
price
50     4.584652
60     3.962450
80     3.836917
70     3.771422
100    3.558563
         ...   
232    0.005458
531    0.005458
730    0.005458
467    0.005458
422    0.005458
Name: proportion, Length: 465, dtype: float64


#### Codificar correctamente las variables categóricas a numéricas

In [39]:
# One Hot Encoder
print("--- Codificación de variables categóricas ---")
print(f"Features antes del encoding: {X.shape[1]}")

# Usar pandas get_dummies para One-Hot Encoding
X_encoded = pd.get_dummies(X, drop_first=True)
print(f"Features después del encoding: {X_encoded.shape[1]}")

# Codificar también la variable objetivo
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"\n--- Codificación variable objetivo ---")
print(f"Mapeo: {dict(zip(le.classes_, le.transform(le.classes_)))}")


--- Codificación de variables categóricas ---
Features antes del encoding: 14
Features después del encoding: 23101

--- Codificación variable objetivo ---
Mapeo: {np.int64(6400): np.int64(0), np.int64(23986): np.int64(1), np.int64(28300): np.int64(2), np.int64(32119): np.int64(3), np.int64(32649): np.int64(4), np.int64(37256): np.int64(5), np.int64(40470): np.int64(6), np.int64(42732): np.int64(7), np.int64(46536): np.int64(8), np.int64(55055): np.int64(9), np.int64(59226): np.int64(10), np.int64(65142): np.int64(11), np.int64(69749): np.int64(12), np.int64(70591): np.int64(13), np.int64(73892): np.int64(14), np.int64(74835): np.int64(15), np.int64(76671): np.int64(16), np.int64(77958): np.int64(17), np.int64(79696): np.int64(18), np.int64(81202): np.int64(19), np.int64(81908): np.int64(20), np.int64(82227): np.int64(21), np.int64(82635): np.int64(22), np.int64(84741): np.int64(23), np.int64(84903): np.int64(24), np.int64(85798): np.int64(25), np.int64(88130): np.int64(26), np.int64(90